# Use cases

Three lookups, run against the merged database:

1. **per compound** - every target it has been tested against, which sets it
   came from, its main target and its selectivity
2. **per target** - every compound tested against it, the most potent and the
   most selective
3. **per family** - which compounds are available for a protein family

`probe.db` holds all seven staging directories. Rebuild it with:

```bash
rm probe.db && python examples/populate_db.py --lenient
python examples/qc_db.py probe.db
```

Real data is messy. Units and endpoint types vary widely (IC50, Ki, AC50,
Potency, % inhibition, dTm), some targets are assay organisms or metadata
placeholders, and 258 targets typed `protein` carry more than one accession
because a source filed a screening panel as a single protein. Nothing here
cleans that up. It picks a comparable scale per query, says which one it
picked, and shows target names as they were reported.

In [1]:
from pathlib import Path

import pandas as pd

from probedb import ProbeDB
from loader.load import STAGING  # the repository's staging directory

pd.set_option("display.max_colwidth", 44)
pd.set_option("display.width", 170)

DB = STAGING.parent / "probe.db"
if not DB.exists():
    raise FileNotFoundError(
        f"{DB} not found. Build it first:\n"
        f"    python examples/populate_db.py --lenient"
    )

db = ProbeDB(DB)

db.counts()

,table,rows
0,compound,16573
1,chembl,15219
2,compound_set,7
3,compound_set_member,20582
4,uniprot,6068
5,target,6908
6,target_uniprot,8584
7,bioactivity_source,51228
8,bioactivity_group,268911
9,bioactivity,457060


## Use case 1: per compound

For each compound:

- which targets it has been measured against
- **which sets it came from** - the staging directories that declared it, which
  is the provenance question. `source_db` below is a different axis: who
  produced the number, not which library the compound belongs to
- its **main target** - strongest potency on a comparable scale
- its **selectivity** - how much weaker the next best target is on that same
  scale

Censored measurements (`>`, `<`) are shown separately, never ranked: `IC50 >
10 uM` says a compound is inactive, not that it is weakly active at 10 uM.

In [2]:
POTENCY_TYPES = {"IC50", "EC50", "AC50", "Ki", "Kd", "GI50", "CC50", "ED50", "Potency"}
TOP_N = 10          # how many rows to print per section
FEATURED = 5        # how many compounds and targets to profile


def compound_label(db, inchikey):
    # name first, then a ChEMBL id, then the key itself: always something
    # identifiable, because not every source names its compounds
    name = db.one("SELECT name FROM compound WHERE inchikey = ?", inchikey)
    chembl_id = db.one("SELECT chembl_id FROM chembl WHERE inchikey = ?", inchikey)
    return name or chembl_id or inchikey


def numeric_potency_rows(hits):
    # a number is only comparable if it is a potency, exact, and on a named
    # scale. 6374 rows in this database have no unit at all
    return hits[
        hits.bioactivity_type.isin(POTENCY_TYPES)
        & (hits.relation == "=")
        & hits.value.notna()
        & hits.unit.notna()
        & ~hits.unit.isin(["", "unspecified"])
    ]


def comparable_scale(numeric):
    """The (endpoint, unit) most of these rows share, or None."""
    if numeric.empty:
        return None
    return numeric.groupby(["bioactivity_type", "unit"]).size().idxmax()


def compound_profile(db, compound):
    hits = db.bioactivities(compound=compound)

    targets = hits[["target_type", "target"]].drop_duplicates().reset_index(drop=True)
    sources = sorted(hits["source_db"].dropna().unique())

    numeric = numeric_potency_rows(hits)
    scale = comparable_scale(numeric)
    potency = pd.DataFrame(columns=["target", "target_type", "value"])
    if scale is not None:
        comparable = numeric[
            (numeric.bioactivity_type == scale[0]) & (numeric.unit == scale[1])
        ]
        potency = (
            comparable.groupby(["target", "target_type"], as_index=False)["value"]
            .median()
            .sort_values("value")
            .reset_index(drop=True)
        )

    counter_screens = hits[hits.relation.isin([">", ">=", "<", "<="])]

    return targets, sources, scale, potency, counter_screens

In [3]:
# the compounds with the most on file, straight out of SQL rather than by
# pulling 457k rows into pandas to count them
busiest = db.read(
    """
    SELECT inchikey, COUNT(*) AS measurements, COUNT(DISTINCT target_id) AS targets
      FROM bioactivity GROUP BY inchikey
     ORDER BY measurements DESC LIMIT ?
""",
    FEATURED,
)

profiled = db.one("SELECT COUNT(DISTINCT inchikey) FROM bioactivity")
print(f"{profiled} of {db.one('SELECT COUNT(*) FROM compound')} compounds have "
      f"a measurement on file")
print(f"profiling the {len(busiest)} with the most\n")

for inchikey in busiest["inchikey"]:
    label = compound_label(db, inchikey)
    targets, sources, scale, potency, counter_screens = compound_profile(db, inchikey)

    print(f"== {label} ==")
    print(f"targets measured: {len(targets)}")

    sets = db.compound_sets(inchikey)["name"].tolist()
    print(f"in sets: {', '.join(sets) if sets else 'none'}")
    print(f"numbers came from: {', '.join(sources) if sources else 'no source recorded'}")

    if potency.empty:
        print("main target: nothing on a comparable potency scale")
    else:
        btype, unit = scale
        best = potency.iloc[0]
        print(f"main target ({btype}, {unit}, {len(potency)} targets on this scale): "
              f"{best.target}  ({best.value:g} {unit})")
        if len(potency) > 1:
            second = potency.iloc[1]
            print(f"selectivity: {second.value / best.value:.1f}-fold vs "
                  f"{second.target} ({second.value:g} {unit}), the next best "
                  f"on the same scale")
        else:
            print(f"selectivity: only one target has comparable {btype} data")

    if not counter_screens.empty:
        print(f"censored: {len(counter_screens)} rows, read qualitatively, "
              f"never ranked")

    print()

12939 of 16573 compounds have a measurement on file
profiling the 5 with the most

== Acetazolamide ==
targets measured: 112
in sets: Novartis_MoA, reFRAME
numbers came from: ChEMBL
main target (Ki, nM, 34 targets on this scale): Carbonic anhydrase  (0.8 nM)
selectivity: 2.2-fold vs Carbonic anhydrase V (1.8 nM), the next best on the same scale
censored: 67 rows, read qualitatively, never ranked

== Vorinostat ==
targets measured: 160
in sets: Novartis_MoA, chemicalprobes, reFRAME, spark
numbers came from: ChEMBL, SPARK-PNAS
main target (IC50, nM, 33 targets on this scale): Histone deacetylase HD1B  (29 nM)
selectivity: 1.1-fold vs HDAC6 (32 nM), the next best on the same scale
censored: 305 rows, read qualitatively, never ranked

== Sunitinib ==
targets measured: 606
in sets: Novartis_MoA, chemicalprobes, reFRAME, spark
numbers came from: ChEMBL, SPARK-PNAS
main target (Kd, nM, 272 targets on this scale): PDGFRB  (0.1375 nM)
selectivity: 3.0-fold vs KIT (0.41 nM), the next best on the

== QUIZARTINIB ==
targets measured: 507
in sets: Probes_n_Drugs, chemicalprobes, reFRAME, spark
numbers came from: ChEMBL, Chemical Probes Portal, Probes & Drugs, SPARK-PNAS
main target (Kd, nM, 68 targets on this scale): FLT3  (4.1 nM)
selectivity: 1.5-fold vs KIT (6 nM), the next best on the same scale
censored: 1611 rows, read qualitatively, never ranked



## Use case 2: per target

The mirror image:

- how many compounds were tested, and which sets they came from
- **most potent** on a comparable scale
- **most selective** - the compound for which this target beats its own next
  best target by the widest margin

One caveat on the second. Ranking every compound by selectivity means building
a full profile for each, and the busiest target here has 3115 of them. So this
looks at the most potent `SELECTIVITY_N` and reports the most selective among
those. That is a different question from "the most selective overall", and it
is the more useful one: a selective compound that does not work is not a lead.

In [4]:
SELECTIVITY_N = 10   # profile this many of the most potent, see the note above


def target_profile(db, target_id):
    hits = db.bioactivities(target=target_id)

    compound_keys = sorted(hits["inchikey"].unique())
    sources = sorted(hits["source_db"].dropna().unique())

    numeric = numeric_potency_rows(hits)
    scale = comparable_scale(numeric)
    potency = pd.DataFrame(columns=["inchikey", "value"])
    if scale is not None:
        comparable = numeric[
            (numeric.bioactivity_type == scale[0]) & (numeric.unit == scale[1])
        ]
        potency = (
            comparable.groupby("inchikey", as_index=False)["value"]
            .median()
            .sort_values("value")
            .reset_index(drop=True)
        )

    return compound_keys, sources, scale, potency


def target_preference(db, compound, target_name):
    # how this target compares to the compound's own best *other* target,
    # reusing the ranking from use case 1
    _, _, _, potency, _ = compound_profile(db, compound)
    at_target = potency[potency.target == target_name]
    others = potency[potency.target != target_name]
    if at_target.empty or others.empty:
        return None
    return others.value.min() / at_target.value.iloc[0]

In [5]:
busiest_targets = db.read(
    """
    SELECT b.target_id, t.type, t.name, COUNT(*) AS measurements,
           COUNT(DISTINCT b.inchikey) AS compounds
      FROM bioactivity b JOIN target t ON t.target_id = b.target_id
     GROUP BY b.target_id ORDER BY measurements DESC LIMIT ?
""",
    FEATURED,
)

measured = db.one("SELECT COUNT(DISTINCT target_id) FROM bioactivity")
print(f"{measured} of {db.one('SELECT COUNT(*) FROM target')} targets have a "
      f"measurement on file")
print(f"profiling the {len(busiest_targets)} with the most\n")

for row in busiest_targets.itertuples():
    compound_keys, sources, scale, potency = target_profile(db, int(row.target_id))

    print(f"== {row.name} ({row.type}) ==")
    print(f"compounds tested: {len(compound_keys)}")

    sets = db.read(
        """
        SELECT s.name, COUNT(DISTINCT m.inchikey) AS n
          FROM compound_set_member m JOIN compound_set s ON s.set_id = m.set_id
         WHERE m.inchikey IN (SELECT DISTINCT inchikey FROM bioactivity
                               WHERE target_id = ?)
         GROUP BY s.name ORDER BY n DESC
    """,
        int(row.target_id),
    )
    print("from sets: " + ", ".join(f"{r.name} ({r.n})" for r in sets.itertuples()))
    print(f"numbers came from: {', '.join(sources) if sources else 'no source recorded'}")

    if potency.empty:
        print("most potent: nothing on a comparable potency scale")
    else:
        btype, unit = scale
        best = potency.iloc[0]
        print(f"most potent ({btype}, {unit}): "
              f"{compound_label(db, best.inchikey)}  ({best.value:g} {unit})")

        ratios = [
            (compound_label(db, k), target_preference(db, k, row.name))
            for k in potency["inchikey"].head(SELECTIVITY_N)
        ]
        ratios = [(c, r) for c, r in ratios if r is not None]
        if not ratios:
            print("most selective: no comparable second target to measure against")
        else:
            top_compound, top_ratio = max(ratios, key=lambda cr: cr[1])
            if top_ratio > 1:
                print(f"most selective of the {len(ratios)} most potent: "
                      f"{top_compound}  ({top_ratio:.1f}-fold vs its next best target)")
            else:
                print(f"most selective: none of the {len(ratios)} most potent is "
                      f"selective for this target. even {top_compound} is "
                      f"{1 / top_ratio:.1f}-fold better somewhere else")

    print()

6304 of 6908 targets have a measurement on file
profiling the 5 with the most



== HDAC6 (protein) ==
compounds tested: 3115


from sets: reFRAME (3107), Novartis_MoA (784), spark (393), Probes_n_Drugs (219), chemicalprobes (196), EUbOPEN (92), opnme (4)
numbers came from: ChEMBL, Chemical Probes Portal, SPARK-PNAS
most potent (IC50, nM): Dactolisib  (2 nM)


most selective of the 9 most potent: Acy-241  (13.5-fold vs its next best target)



== Replicase polyprotein 1ab (protein) ==
compounds tested: 3096


...

== EGFR (protein) ==
compounds tested: 476
from sets: reFRAME (460), Novartis_MoA (167), spark (126), chemicalprobes (122), Probes_n_Drugs (96), EUbOPEN (57), opnme (2)
numbers came from: ChEMBL, Chemical Probes Portal, EUbOPEN, SPARK-PNAS


most potent (IC50, nM): IMATINIB  (0.11 nM)


most selective of the 8 most potent: AV-412 free base  (34.5-fold vs its next best target)



== TDP1 (protein) ==
compounds tested: 1196


from sets: reFRAME (1193), Novartis_MoA (352), spark (186), Probes_n_Drugs (71), chemicalprobes (53), EUbOPEN (26), opnme (1)
numbers came from: ChEMBL, SPARK-PNAS
most potent (Potency, nM): Carbetapentane  (1 nM)
most selective of the 4 most potent: Carbetapentane  (316.2-fold vs its next best target)



## Use case 3: per family

"Family" means two different things here, and they are worth keeping apart.

**A target of type `family`** is our own curated grouping, several accessions
under one name that a source measured together, like `PARP 1, 2 and 3`. It is
a target: it has its own measurements, because an assay was run against the
group.

**`uniprot.superfamily`** is UniProt's classification of a single protein, read
from `reference/uniprot_protein_families.tsv` so every source agrees on it.
1883 of 6068 accessions have one, and those carry 71% of the measurements.

The old version of this notebook had neither and fell back to the template.

In [6]:
# the curated ones: a family is a target, so it has measurements of its own
curated = db.read("""
  SELECT t.target_id, t.name, COUNT(*) AS members,
         (SELECT COUNT(*) FROM bioactivity b WHERE b.target_id = t.target_id) AS measurements
    FROM target t JOIN target_uniprot tu ON tu.target_id = t.target_id
   WHERE t.type = 'family'
   GROUP BY t.target_id ORDER BY measurements DESC
""")

print(f"{len(curated)} curated family targets\n")
curated.head(TOP_N)

224 curated family targets



,target_id,name,members,measurements
0,3223,Adrenergic receptor alpha-1,3,452
1,3021,Histone deacetylase,11,405
2,3264,Adrenergic receptor alpha-2,3,305
3,3596,Muscarinic acetylcholine receptor,5,304
4,3965,Phosphodiesterase 4,4,232
5,3303,Sodium channel alpha subunits; brain (Ty...,3,230
6,3310,Serotonin 2 (5-HT2) receptor,3,194
7,3169,Phosphodiesterase 3,2,186
8,3428,Estrogen receptor,2,179
9,3273,Adrenergic receptor alpha-2,3,136


In [7]:
# the classification: db.families() splits the hierarchy into levels and
# family= filters targets(), compounds() and bioactivities() by one of them
db.families(like="kinase").head(TOP_N)

,family,proteins,targets
0,Protein kinase superfamily,249,474
1,Tyr protein kinase family,58,120
2,AGC Ser/Thr protein kinase family,37,61
3,CAMK Ser/Thr protein kinase family,34,51
4,Ser/Thr protein kinase family,34,52
5,CMGC Ser/Thr protein kinase family,33,107
6,STE Ser/Thr protein kinase family,26,34
7,TKL Ser/Thr protein kinase family,20,35
8,MAP kinase subfamily,13,29
9,PI3/PI4-kinase family,13,34


In [8]:
# "for the RAS family, what is available?"
FAMILY = "Ras family"

# a family answer is only as good as the target rows underneath it. drop the
# protein targets holding more than one accession first: those are screening
# panels filed as a single protein, and every accession in them drags the
# target into its own family. a complex with several members is fine
members = db.targets(family=FAMILY)
per_target = members.groupby(["target_id", "type"], as_index=False).uniprot_id.nunique()
lumped = per_target[(per_target.type == "protein") & (per_target.uniprot_id > 1)].target_id
print(f"dropping {len(lumped)} lumped target(s): "
      f"{', '.join(members[members.target_id.isin(lumped)].name.unique())}\n")

rows = db.bioactivities(family=FAMILY)
rows = rows[~rows.target_id.isin(lumped)]

numeric = numeric_potency_rows(rows)
scale = comparable_scale(numeric)
btype, unit = scale
print(f"ranking on {btype} in {unit}\n")

available = (
    numeric[(numeric.bioactivity_type == btype) & (numeric.unit == unit)]
    .groupby(["inchikey", "compound", "target"], as_index=False)
    .agg(n=("value", "size"), best=("value", "min"))
    .merge(db.compounds()[["inchikey", "sets"]], on="inchikey")
    .sort_values("best")
)

available.head(TOP_N)[["compound", "target", "n", "best", "sets"]]

dropping 2 lumped target(s): Protein cereblon, GTPase KRas

ranking on IC50 in nM



,compound,target,n,best,sets
9,Adagrasib,KRAS,6,5.00,"Probes_n_Drugs, chemicalprobes, reFRAME"
2,BI-0474,KRAS,1,7.00,"Probes_n_Drugs, opnme"
8,BMS-214662,KRAS,3,8.40,reFRAME
6,sotorasib,KRAS,5,9.66,"Probes_n_Drugs, chemicalprobes, reFRAME"
0,opnurasib,KRAS,6,10.00,reFRAME
7,sotorasib,SOS1-KRAS,1,15.80,"Probes_n_Drugs, chemicalprobes, reFRAME"
3,LONAFARNIB,KRAS,1,40.00,"Probes_n_Drugs, reFRAME, spark"
10,Adagrasib,SOS1-KRAS,1,130.00,"Probes_n_Drugs, chemicalprobes, reFRAME"
5,DP-4978,KRAS,1,150.00,reFRAME
11,GDC-6036,KRAS,1,600.00,reFRAME


In [9]:
# the sets column above is the answer to "which sets are these derived from".
# what it cannot yet answer is "which of them are chemical probes, which are
# chemogenomic compounds, which are drugs": compound_set.category exists and
# every set is currently 'library', because the category is derived from the
# staging directory and nobody has curated it
db.sets()

,set_id,name,category,source_db,compounds
0,6,reFRAME,library,reFRAME,8513
1,3,Probes_n_Drugs,library,Probes_n_Drugs,4840
2,2,Novartis_MoA,library,Novartis_MoA,4185
3,4,chemicalprobes,library,chemicalprobes,1223
4,7,spark,library,spark,987
5,1,EUbOPEN,library,EUbOPEN,732
6,5,opnme,library,opnme,102


## A closing warning: the top of a ranking is not a fact

Use case 2 reports Imatinib as the most potent EGFR compound on file, at
0.11 nM. Imatinib is a BCR-ABL and KIT inhibitor. It is not a sub-nanomolar
EGFR inhibitor, and the database says so itself, twice over, in the same two
rows the ranking drew from.

Nothing is averaged or deduplicated on load, so both survive and you can see
the disagreement. Had the loader merged them into one number, the conflict
would be gone and only the wrong answer would be left.

In [10]:
db.bioactivities(compound="IMATINIB", target="EGFR")[
    ["bioactivity_type", "relation", "value", "unit", "assay_description",
     "source", "source_url"]
]

,bioactivity_type,relation,value,unit,assay_description,source,source_url
0,IC50,>,100000.00,nM,Inhibition of the epidermal growth facto...,ChEMBL 37,https://doi.org/10.1016/S0960-894X(96)00...
1,Kd,>,10000.00,nM,Binding constant for EGFR kinase domain,PMID:18183025,https://doi.org/10.1038/nbt1358
2,Kd,=,7600.00,nM,"Binding constant for EGFR(L747-E749del, ...",PMID:18183025,https://doi.org/10.1038/nbt1358
3,Inhibition,=,14.00,%,Inhibition of human EGFR activity at 10 ...,PMID:16415863,https://doi.org/10.1038/nchembio760
4,Inhibition,=,0.00,%,Inhibition of HER-1 activity at 10 uM by...,PMID:16415863,https://doi.org/10.1038/nchembio760
5,Kd,>,10000.00,nM,Binding constant for EGFR kinase domain,PMID:22037378,https://doi.org/10.1038/nbt.1990
6,Kd,=,7600.00,nM,"Binding constant for EGFR(L747-E749del, ...",PMID:22037378,https://doi.org/10.1038/nbt.1990
7,Ki,>,1995.26,nM,PUBCHEM_BIOASSAY: Navigating the Kinome....,ChEMBL 37,NaN
8,Kd,>,30000.00,nM,"Kinobeads (epsilon), multiple immobilize...",PMID:29191878,https://doi.org/10.1126/science.aan4368
9,Inhibition,=,13.00,%,Inhibition of EGFR (unknown origin) at 1...,PMID:28916158,https://doi.org/10.1016/j.bmc.2017.08.039
